# 📊 Notebook 05: Evaluation & Comparison

Notebook này thực hiện:
1. Đánh giá RAGAS cho từng model
2. So sánh 3 models (Vistral vs PhoGPT vs Qwen)
3. So sánh RAG vs Non-RAG
4. So sánh embedding models (PhoBERT vs E5)
5. Export reports

In [ ]:
import os, sys
PROJECT_DIR = '/content/vietnamese-legal-qa'
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

from src.config.settings import Settings
from src.components.evaluator import Evaluator
from src.services.evaluation_service import EvaluationService
# ... (initialize chat_service as in notebook 04)

settings = Settings.load('config.yaml')
evaluator = Evaluator(settings)
print('Ready for evaluation!')

In [ ]:
# === Prepare test set ===
from src.components.data_collector import DataCollector
collector = DataCollector(settings)
qa_pairs = collector.load_alqac_dataset('data/alqac')

# Use 10% as test set
import random
random.seed(42)
test_pairs = random.sample(qa_pairs, min(50, len(qa_pairs)))
print(f'Test set: {len(test_pairs)} samples')

In [ ]:
# === Evaluate single model ===
# (Assuming chat_service is initialized with qwen model)

questions = [p.question for p in test_pairs]
ground_truths = [p.answer for p in test_pairs]

# Get answers from system
session_id = chat_service.new_session()
answers = []
contexts_list = []

for pair in test_pairs:
    response = chat_service.chat(pair.question, session_id, 'qwen')
    answers.append(response.answer)
    contexts_list.append([doc.content for doc in response.sources])

# RAGAS evaluation
report = evaluator.evaluate_batch(
    questions=questions,
    answers=answers,
    contexts_list=contexts_list,
    ground_truths=ground_truths,
    model_name='qwen_finetuned'
)

print(f'\n📊 RAGAS Results (Qwen2.5-7B fine-tuned):')
print(f'  Faithfulness:      {report.avg_faithfulness:.4f}')
print(f'  Context Recall:    {report.avg_context_recall:.4f}')
print(f'  Answer Relevancy:  {report.avg_answer_relevancy:.4f}')

In [ ]:
# === RAG vs Non-RAG Comparison (US-20) ===
eval_service = EvaluationService(settings, chat_service, evaluator)

rag_comparison = eval_service.run_rag_comparison(
    test_pairs=test_pairs[:20],  # Subset for speed
    model_key='qwen',
    session_id=session_id
)

if 'rag' in rag_comparison and 'no_rag' in rag_comparison:
    comparison_table = evaluator.compare_rag_vs_no_rag(
        rag_comparison['rag'],
        rag_comparison['no_rag']
    )
    print(comparison_table)

In [ ]:
# === Export reports ===
evaluator.export_report(report, 'qwen_evaluation.json')
print('\n✅ Reports exported to evaluation/reports/')
print('\nDone! Use these results in your thesis/report.')